# Delhi Metro Operations — ML Modeling
### Hourly Demand Forecasting (Regression) + Station Load Clustering

This notebook extends the Delhi Metro GTFS EDA with two ML models:
1. **Demand Forecasting** — predicts hourly stop-visit volume per route using a Random Forest Regressor
2. **Station Load Clustering** — segments all 262 stations into operational load tiers using KMeans

Data: official DMRC GTFS feed (`stop_times.txt`, `trips.txt`, `stops.txt`, `routes.txt`, `calendar.txt`)


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, silhouette_score
from sklearn.model_selection import train_test_split, GroupKFold

calendar = pd.read_csv('calendar.txt')
routes = pd.read_csv('routes.txt')
stop_times = pd.read_csv('stop_times.txt')
stops = pd.read_csv('stops.txt')
trips = pd.read_csv('trips.txt')

print('Routes:', routes.shape, '| Stop Times:', stop_times.shape, '| Stops:', stops.shape, '| Trips:', trips.shape)


Routes: (36, 12) | Stop Times: (128434, 12) | Stops: (262, 6) | Trips: (5438, 10)


## 1. Merge GTFS tables & extract hour-of-day

In [2]:
trip_stop_times = pd.merge(stop_times, trips, on="trip_id")
full_df = pd.merge(trip_stop_times, stops, on="stop_id")
full_df = pd.merge(full_df, routes, on="route_id")

full_df['hour'] = pd.to_datetime(full_df['arrival_time'], format='%H:%M:%S', errors='coerce').dt.hour
full_df = full_df[(full_df['hour'] >= 0) & (full_df['hour'] <= 23)]

print('Merged dataframe shape:', full_df.shape)
full_df[['route_long_name','stop_name','arrival_time','hour','service_id']].head()


Merged dataframe shape: (127043, 38)


,route_long_name,stop_name,arrival_time,hour,service_id
0,RED_Rithala to Dilshad Garden,Rithala,05:28:08,5.0,weekday
1,RED_Rithala to Dilshad Garden,Rohini West,05:30:58,5.0,weekday
2,RED_Rithala to Dilshad Garden,Rohini East,05:33:28,5.0,weekday
3,RED_Rithala to Dilshad Garden,Pitampura,05:35:33,5.0,weekday
4,RED_Rithala to Dilshad Garden,Kohat Enclave,05:37:53,5.0,weekday


## 2. Demand Forecasting Model

**Goal:** Predict hourly stop-visit volume for a given route, to power proactive frequency/scheduling decisions instead of static timetables.

We aggregate to (route, service_type, hour) granularity — each row is "how many times did trains on this route stop somewhere, in this hour, on a weekday/weekend." This is the natural prediction target for a scheduling system.

**Two evaluation protocols, reported separately for honesty:**
- **Experiment A (random split):** model has seen each route's average demand level before — this is the realistic deployment scenario for a metro that already has historical data on every one of its existing routes.
- **Experiment B (Group K-Fold by route):** model is tested on routes it has *never* seen — a much harder, stricter generalization test.

In [3]:
demand = full_df.groupby(['route_id','route_long_name','service_id','hour']).size().reset_index(name='stop_visits')

demand['is_weekday'] = (demand['service_id'] == 'weekday').astype(int)
demand['is_peak'] = demand['hour'].isin([8,9,17,18]).astype(int)
demand['hour_sin'] = np.sin(2*np.pi*demand['hour']/24)
demand['hour_cos'] = np.cos(2*np.pi*demand['hour']/24)

route_stop_counts = full_df.groupby('route_id')['stop_id'].nunique().rename('num_stops_on_route')
demand = demand.merge(route_stop_counts, on='route_id')

print('Demand table shape:', demand.shape)
print(f"stop_visits — mean={demand['stop_visits'].mean():.1f}, std={demand['stop_visits'].std():.1f}, "
      f"min={demand['stop_visits'].min()}, max={demand['stop_visits'].max()}")
demand.head()


Demand table shape: (622, 10)
stop_visits — mean=204.2, std=143.8, min=2, max=554


,route_id,route_long_name,service_id,hour,stop_visits,is_weekday,is_peak,hour_sin,hour_cos,num_stops_on_route
0,0,RED_Rithala to Dilshad Garden,weekday,5.0,24,1,0,0.965926,2.588190e-01,21
1,0,RED_Rithala to Dilshad Garden,weekday,6.0,27,1,0,1.000000,6.123234e-17,21
2,0,RED_Rithala to Dilshad Garden,weekday,7.0,12,1,0,0.965926,-2.588190e-01,21
3,0,RED_Rithala to Dilshad Garden,weekday,16.0,7,1,0,-0.866025,-5.000000e-01,21
4,0,RED_Rithala to Dilshad Garden,weekday,17.0,151,1,1,-0.965926,-2.588190e-01,21


In [4]:
# ---- Experiment A: random split, route_avg_demand included (realistic: known routes) ----
route_avg = demand.groupby('route_id')['stop_visits'].mean().rename('route_avg_demand')
demand_a = demand.merge(route_avg, on='route_id')

features_a = ['hour','hour_sin','hour_cos','is_weekday','is_peak','route_avg_demand','num_stops_on_route']
X_a, y_a = demand_a[features_a], demand_a['stop_visits']
Xa_tr, Xa_te, ya_tr, ya_te = train_test_split(X_a, y_a, test_size=0.2, random_state=42)

baseline_pred = np.full_like(ya_te, ya_tr.mean(), dtype=float)
baseline_mae = mean_absolute_error(ya_te, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(ya_te, baseline_pred))

lr = LinearRegression().fit(Xa_tr, ya_tr)
lr_pred = lr.predict(Xa_te)

rf_a = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42, n_jobs=-1).fit(Xa_tr, ya_tr)
rf_pred = rf_a.predict(Xa_te)

print("=== Experiment A: Random split ===")
print(f"Baseline (mean):    MAE={baseline_mae:.2f}  RMSE={baseline_rmse:.2f}")
print(f"Linear Regression:  MAE={mean_absolute_error(ya_te,lr_pred):.2f}  RMSE={np.sqrt(mean_squared_error(ya_te,lr_pred)):.2f}  R2={r2_score(ya_te,lr_pred):.4f}")
print(f"Random Forest:      MAE={mean_absolute_error(ya_te,rf_pred):.2f}  RMSE={np.sqrt(mean_squared_error(ya_te,rf_pred)):.2f}  R2={r2_score(ya_te,rf_pred):.4f}")
print(f"\nRF reduces MAE by {(1-mean_absolute_error(ya_te,rf_pred)/baseline_mae)*100:.1f}% vs. naive mean baseline")


=== Experiment A: Random split ===
Baseline (mean):    MAE=124.22  RMSE=153.06
Linear Regression:  MAE=35.18  RMSE=49.17  R2=0.8968
Random Forest:      MAE=17.58  RMSE=27.15  R2=0.9685

RF reduces MAE by 85.8% vs. naive mean baseline


In [5]:
# ---- Experiment B: 5-fold Group CV by route_id (strict: unseen routes) ----
features_b = ['hour','hour_sin','hour_cos','is_weekday','is_peak','num_stops_on_route']
X_b, y_b, groups = demand[features_b], demand['stop_visits'], demand['route_id']

gkf = GroupKFold(n_splits=5)
maes, rmses, r2s = [], [], []
for train_idx, test_idx in gkf.split(X_b, y_b, groups):
    m = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42, n_jobs=-1)
    m.fit(X_b.iloc[train_idx], y_b.iloc[train_idx])
    p = m.predict(X_b.iloc[test_idx])
    maes.append(mean_absolute_error(y_b.iloc[test_idx], p))
    rmses.append(np.sqrt(mean_squared_error(y_b.iloc[test_idx], p)))
    r2s.append(r2_score(y_b.iloc[test_idx], p))

print("=== Experiment B: 5-fold Group-CV (unseen routes) ===")
print(f"RF:  MAE={np.mean(maes):.2f} (+/-{np.std(maes):.2f})  RMSE={np.mean(rmses):.2f} (+/-{np.std(rmses):.2f})  R2={np.mean(r2s):.4f}")


=== Experiment B: 5-fold Group-CV (unseen routes) ===
RF:  MAE=34.88 (+/-14.48)  RMSE=50.78 (+/-16.73)  R2=0.7930


In [6]:
importances = sorted(zip(features_a, rf_a.feature_importances_), key=lambda x: -x[1])
print("Feature importances (Experiment A model):")
for f, imp in importances:
    print(f"  {f:20s} {imp:.3f}")


Feature importances (Experiment A model):
  route_avg_demand     0.731
  num_stops_on_route   0.137
  hour                 0.103
  hour_sin             0.015
  hour_cos             0.012
  is_peak              0.002
  is_weekday           0.000


## 3. Station Load Clustering

**Goal:** Segment all 262 stations into operational load tiers (rather than relying on a manual "top-15 busiest" list), to systematically flag which stations need infrastructure upgrades, dwell-time increases, or staffing changes.

Features per station: total stop-visits, number of routes serving it (interchange-ness), and peak-hour share of its traffic.

In [7]:
station_visits = full_df.groupby('stop_name').size().rename('total_visits')
station_peak = full_df[full_df['hour'].isin([8,9,17,18])].groupby('stop_name').size().rename('peak_visits')
station_routes = full_df.groupby('stop_name')['route_id'].nunique().rename('num_routes')

station_df = pd.concat([station_visits, station_peak, station_routes], axis=1).fillna(0)
station_df['peak_ratio'] = (station_df['peak_visits'] / station_df['total_visits'].replace(0, np.nan)).fillna(0)

features = ['total_visits','num_routes','peak_ratio']
X = station_df[features]
X_scaled = StandardScaler().fit_transform(X)

print("Station feature table:", station_df.shape)
station_df.head()


Station feature table: (262, 4)


,total_visits,peak_visits,num_routes,peak_ratio
stop_name,,,,
AIIMS,731,145,4,0.198358
Adarsh Nagar,246,48,2,0.195122
Akshardham,363,85,2,0.234160
Alpha 1,448,124,2,0.276786
Anand Vihar,717,171,4,0.238494


In [8]:
print("Silhouette score by k:")
for k in range(2, 8):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    print(f"  k={k}: silhouette={silhouette_score(X_scaled, labels):.4f}")


Silhouette score by k:
  k=2: silhouette=0.5136
  k=3: silhouette=0.5004
  k=4: silhouette=0.5372


  k=5: silhouette=0.5392
  k=6: silhouette=0.5537


  k=7: silhouette=0.5642


In [9]:
# k=4 chosen for interpretability: maps cleanly to a 4-tier operational classification
k = 4
km = KMeans(n_clusters=k, random_state=42, n_init=10)
station_df['cluster'] = km.fit_predict(X_scaled)
sil = silhouette_score(X_scaled, station_df['cluster'])
print(f"Final model: k={k}, silhouette_score={sil:.4f}")

profile = station_df.groupby('cluster')[['total_visits','num_routes','peak_ratio']].mean().round(2)
profile['num_stations'] = station_df.groupby('cluster').size()
profile = profile.sort_values('total_visits', ascending=False)
profile


Final model: k=4, silhouette_score=0.5372


,total_visits,num_routes,peak_ratio,num_stations
cluster,,,,
3,1105.56,6.75,0.22,16
2,717.25,4.00,0.14,8
1,575.41,4.06,0.24,104
0,326.66,2.00,0.24,134


In [10]:
order = profile.index.tolist()
labels_map = {order[0]: 'Tier 1 - Major Interchange', order[1]: 'Tier 2 - High Load',
              order[2]: 'Tier 3 - Moderate', order[3]: 'Tier 4 - Low Load'}
station_df['tier'] = station_df['cluster'].map(labels_map)

print(station_df['tier'].value_counts())
print()
print("Tier 1 (Major Interchange) stations:")
station_df[station_df['tier']=='Tier 1 - Major Interchange'].sort_values('total_visits', ascending=False)[['total_visits','num_routes','peak_ratio']]


tier
Tier 4 - Low Load             134
Tier 3 - Moderate             104
Tier 1 - Major Interchange     16
Tier 2 - High Load              8
Name: count, dtype: int64

Tier 1 (Major Interchange) stations:


,total_visits,num_routes,peak_ratio
stop_name,,,
Kashmere Gate,1640,12,0.231098
Rajiv Chowk,1455,8,0.217182
Central Secretariat,1257,8,0.219570
Mandi House,1255,8,0.236653
Qutab Minar,1200,6,0.163333
Dilli Haat - INA,1091,6,0.210816
Sikanderpur,1090,6,0.178899
Hauz Khas,1084,6,0.214945
Rajouri Garden,1080,6,0.236111


## Summary

| Model | Metric | Result |
|---|---|---|
| Demand Forecasting (Random Forest, known routes) | MAE / RMSE / R² | 17.6 / 27.2 / 0.97 |
| Demand Forecasting (Random Forest, unseen routes, 5-fold Group-CV) | MAE / RMSE / R² | 34.9 / 50.8 / 0.79 |
| Station Load Clustering (KMeans, k=4) | Silhouette Score | 0.537 |
| Station Load Clustering | Stations identified as Tier 1 (Major Interchange) | 16 / 262 |
